# EDA General — Análisis Exploratorio Profundo de los 3 Datasets
**Machine Learning I — Universidad Externado de Colombia**

---

Este cuaderno realiza el análisis exploratorio completo de los tres datasets del Taller 2: `matches.csv`, `players.csv` y `xg_train.csv`. El objetivo es entender la naturaleza estadística de los datos, detectar patrones, correlaciones y señales cruzadas entre fuentes antes de cualquier modelado.

| Dataset | Descripción |
|---|---|
| `matches.csv` | Partidos de Premier League — resultados, cuotas, estadísticas de partido |
| `players.csv` | Catálogo FPL — 822 jugadores con métricas de Fantasy Premier League |
| `xg_train.csv` | Disparos — 7,198 tiros con coordenadas, contexto y variable `is_goal` |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, chi2_contingency, kruskal
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import warnings
warnings.filterwarnings('ignore')

# Identidad Visual Premier League
PL_PURPLE = '#38003c'
PL_CYAN   = '#00ff85'
PL_PINK   = '#e90052'
PL_WHITE  = '#ffffff'
sns.set_theme(style='darkgrid')


---
## SECCIÓN 1: Inventario de Datasets

Cargamos los tres datasets y mostramos su estructura, tamaño y estadísticas básicas.


In [ ]:
# Carga de los tres datasets
matches  = pd.read_csv('../../data/matches.csv')
players  = pd.read_csv('../../data/players.csv', skipinitialspace=True)
players.columns = players.columns.str.strip()
xg       = pd.read_csv('../../data/xg_train.csv')
events   = pd.read_csv('../../data/events.csv')

matches['date_dt'] = pd.to_datetime(matches['date'], dayfirst=True, errors='coerce')
matches = matches.dropna(subset=['date_dt']).sort_values('date_dt').reset_index(drop=True)

print('='*70)
print('INVENTARIO DE DATASETS')
print('='*70)
print(f'  matches.csv  → {len(matches):>6,} partidos  | {matches.shape[1]} columnas')
print(f'  players.csv  → {len(players):>6,} jugadores | {players.shape[1]} columnas')
print(f'  xg_train.csv → {len(xg):>6,} disparos  | {xg.shape[1]} columnas')
print(f'  events.csv   → {len(events):>6,} eventos   | {events.shape[1]} columnas')
print()

vc = matches['ftr'].value_counts()
n  = len(matches)
print('='*70)
print('DISTRIBUCIÓN DEL TARGET — ftr (Full Time Result)')
print('='*70)
print(f'  H  (Victoria Local)    : {vc["H"]:>3} partidos  ({vc["H"]/n*100:.1f}%)')
print(f'  D  (Empate)            : {vc["D"]:>3} partidos  ({vc["D"]/n*100:.1f}%)')
print(f'  A  (Victoria Visitante): {vc["A"]:>3} partidos  ({vc["A"]/n*100:.1f}%)')
print()
print('='*70)
print('COLUMNAS DISPONIBLES Y CLASIFICACIÓN')
print('='*70)
sep = '  ' + '─'*66
print(sep)
print(f"  {'Variable':<30} {'Disponible antes del partido':^30}")
print(sep)
rows = [
    ('date / home_team / away_team', '✓  Metadatos administrativos'),
    ('referee',                      '✓  Árbitro asignado (semanas antes)'),
    ('b365h / b365d / b365a',        '✓  Cuotas Bet365 (días antes)'),
    ('avgh / avgd / avga',           '✓  Cuotas promedio multi-casas'),
    ('ftr',                          '—  TARGET — resultado final'),
    ('fthg / ftag',                  '✗  LEAKAGE — goles del partido'),
    ('hthg / htag / htr',           '✗  LEAKAGE — resultado medio tiempo'),
    ('hs / as_ / hst / ast',        '✗  LEAKAGE — tiros totales/al arco'),
    ('hf / af / hc / ac',           '✗  LEAKAGE — faltas y corners'),
    ('hy / ay / hr / ar',           '✗  LEAKAGE — tarjetas'),
]
for var, estado in rows:
    print(f'  {var:<30} {estado}')
print(sep)


In [ ]:
# Estadísticas descriptivas de matches.csv
print('='*70)
print('MATCHES.CSV — ESTADÍSTICAS DESCRIPTIVAS')
print('='*70)
print(f'  Rango temporal: {matches["date_dt"].min().date()} → {matches["date_dt"].max().date()}')
print(f'  Equipos únicos: {matches["home_team"].nunique()}')
print(f'  Árbitros únicos: {matches["referee"].nunique()}')
print()
num_cols = ['fthg','ftag','hs','as_','hst','ast','hf','af','hc','ac','hy','ay']
display(matches[num_cols].describe().round(2))
print()
print('='*70)
print('PLAYERS.CSV — ESTADÍSTICAS DESCRIPTIVAS')
print('='*70)
print(f'  Jugadores únicos : {players["player_name"].nunique()}')
print(f'  Equipos únicos   : {players["team"].nunique()}')
if 'position' in players.columns:
    print(f'  Posiciones       : {dict(players["position"].value_counts())}')
display(players[['total_points','influence','threat','creativity']].describe().round(2))
print()
print('='*70)
print('XG_TRAIN.CSV — ESTADÍSTICAS DESCRIPTIVAS')
print('='*70)
print(f'  Disparos totales: {len(xg):,}')
print(f'  Goles           : {xg["is_goal"].sum():,} ({xg["is_goal"].mean()*100:.2f}%)')
print(f'  No-goles        : {(~xg["is_goal"].astype(bool)).sum():,} ({(1-xg["is_goal"].mean())*100:.2f}%)')
display(xg[['dist_al_arco','angulo_tiro','is_BigChance','is_Penalty','is_OneOnOne']].describe().round(3))


---
## SECCIÓN 2: Distribución de Resultados y Dinámica Temporal


In [ ]:
# ── EDA 1: Distribución de resultados y desbalance de clases ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1A. Barplot resultados
ax = axes[0]
labels = ['H (Local)', 'D (Empate)', 'A (Visita)']
counts = [vc['H'], vc['D'], vc['A']]
colors = [PL_CYAN, PL_WHITE, PL_PINK]
bars = ax.bar(labels, counts, color=colors, edgecolor=PL_PURPLE, linewidth=1.5)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, cnt + 1,
            f'{cnt}\n({cnt/n*100:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Distribución de Resultados\n(Target: ftr)', color=PL_PURPLE, fontweight='bold')
ax.set_ylabel('Partidos')
ax.set_ylim(0, max(counts)*1.2)

# 1B. Tendencia de resultados en el tiempo
ax = axes[1]
matches['month'] = matches['date_dt'].dt.to_period('M')
monthly = matches.groupby('month')['ftr'].value_counts(normalize=True).unstack(fill_value=0)
months_str = [str(m) for m in monthly.index]
for cls, col in zip(['H','D','A'], [PL_CYAN, PL_WHITE, PL_PINK]):
    if cls in monthly.columns:
        ax.plot(months_str, monthly[cls], color=col, marker='o', ms=5, label=cls, lw=2)
ax.set_title('Tasa de Resultados por Mes', color=PL_PURPLE, fontweight='bold')
ax.set_ylabel('Proporción')
ax.set_xlabel('Mes')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# 1C. Goles totales por resultado
ax = axes[2]
matches['total_goals'] = matches['fthg'] + matches['ftag']
for cls, col, lbl in zip(['H','D','A'], [PL_CYAN, PL_WHITE, PL_PINK],
                         ['Local gana','Empate','Visita gana']):
    data = matches[matches['ftr']==cls]['total_goals']
    ax.hist(data, bins=range(0,11), alpha=0.6, color=col,
            label=f'{lbl} (μ={data.mean():.2f})', edgecolor=PL_PURPLE)
ax.set_title('Goles Totales por Tipo de Resultado', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Goles totales')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=9)

plt.suptitle('EDA 1 — Distribución de Resultados (matches.csv)', fontsize=14,
             fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo3_eda1_resultados.png', bbox_inches='tight', dpi=150)
plt.show()

print('Test chi-cuadrado — ¿son los resultados uniformes?')
chi2, p_chi, _, _ = chi2_contingency(pd.crosstab(matches['month'].astype(str), matches['ftr']))
print(f'  chi2={chi2:.3f}, p={p_chi:.4f} → {"Distribución varía por mes" if p_chi<0.05 else "Sin variación mensual significativa"}')


---
## SECCIÓN 3: Factor Árbitro


In [ ]:
# ── EDA 2: Factor Árbitro — ¿Favorece el árbitro al equipo local? ──
top_refs = matches['referee'].value_counts().head(10).index
df_refs  = matches[matches['referee'].isin(top_refs)].copy()

ref_stats = df_refs.groupby('referee').apply(
    lambda g: pd.Series({
        'partidos' : len(g),
        'pct_H'    : (g['ftr']=='H').mean()*100,
        'pct_D'    : (g['ftr']=='D').mean()*100,
        'pct_A'    : (g['ftr']=='A').mean()*100,
        'media_tarj': (g['hy']+g['ay']+g['hr']+g['ar']).mean(),
    })
).sort_values('pct_H', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras apiladas de resultados por árbitro
ax = axes[0]
refs = ref_stats.index
x = np.arange(len(refs))
ax.barh(x, ref_stats['pct_H'], color=PL_CYAN, label='H', alpha=0.85)
ax.barh(x, ref_stats['pct_D'], left=ref_stats['pct_H'], color=PL_WHITE, label='D', alpha=0.85)
ax.barh(x, ref_stats['pct_A'],
        left=ref_stats['pct_H']+ref_stats['pct_D'], color=PL_PINK, label='A', alpha=0.85)
ax.axvline(vc['H']/n*100, color=PL_CYAN, linestyle='--', lw=1.5, alpha=0.6, label=f'Media H ({vc["H"]/n*100:.1f}%)')
ax.set_yticks(x); ax.set_yticklabels(refs, fontsize=9)
ax.set_xlabel('% de partidos')
ax.set_title('Distribución de Resultados por Árbitro\n(Top 10 árbitros)', color=PL_PURPLE, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)

# Media de tarjetas por árbitro
ax = axes[1]
bars = ax.bar(refs, ref_stats['media_tarj'], color=PL_PINK, edgecolor=PL_PURPLE)
ax.axhline(ref_stats['media_tarj'].mean(), color=PL_CYAN, linestyle='--', lw=2, label='Media global')
ax.set_title('Media de Tarjetas por Árbitro', color=PL_PURPLE, fontweight='bold')
ax.set_ylabel('Tarjetas promedio por partido')
ax.tick_params(axis='x', rotation=45)
ax.legend()

plt.suptitle('EDA 2 — Factor Árbitro (matches.csv)', fontsize=14,
             fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo3_eda2_arbitro.png', bbox_inches='tight', dpi=150)
plt.show()

# Kruskal-Wallis: ¿varía el % de victorias locales por árbitro?
groups_ref = [group['ftr'].map({'H':1,'D':0,'A':-1}).values
              for _, group in df_refs.groupby('referee')]
stat_kw, p_kw = kruskal(*groups_ref)
print(f'Kruskal-Wallis (resultado vs árbitro): stat={stat_kw:.3f}, p={p_kw:.4f}')
print(f'  → {"El árbitro SÍ influye en el resultado" if p_kw<0.05 else "Sin efecto árbitro significativo"}')
print()
print(ref_stats.round(1).to_string())


---
## SECCIÓN 4: Cuotas Bet365 como Señal Predictiva


In [ ]:
# ── EDA 3: Cuotas Bet365 como señal predictiva ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (cuota, label, color) in zip(axes, [
    ('b365h', 'Cuota Local (b365h)', PL_CYAN),
    ('b365d', 'Cuota Empate (b365d)', PL_WHITE),
    ('b365a', 'Cuota Visita (b365a)', PL_PINK),
]):
    for cls, col in zip(['H','D','A'], [PL_CYAN, PL_WHITE, PL_PINK]):
        data = matches[matches['ftr']==cls][cuota].dropna()
        ax.hist(data, bins=20, alpha=0.5, color=col, label=cls,
                density=True, edgecolor=PL_PURPLE, linewidth=0.5)
    ax.set_title(f'Distribución de {label}\npor Resultado Real', color=PL_PURPLE, fontweight='bold')
    ax.set_xlabel('Cuota')
    ax.set_ylabel('Densidad')
    ax.legend()

plt.suptitle('EDA 3 — Cuotas Bet365 como Señal (matches.csv)', fontsize=14,
             fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo3_eda3_cuotas.png', bbox_inches='tight', dpi=150)
plt.show()

# Spearman de cuotas vs resultado
y_enc = matches['ftr'].map({'H':1,'D':0,'A':-1})
print('Correlaciones Spearman: Cuotas vs Resultado codificado (H=+1, D=0, A=-1)')
print(f"  {'Variable':<12}  {'ρ':>8}  {'p-value':>10}  Veredicto")
print('  ' + '─'*55)
for col in ['b365h', 'b365d', 'b365a']:
    rho, pval = spearmanr(matches[col].dropna(), y_enc[matches[col].notna()])
    verdict = '✓ Significativo' if pval < 0.05 else '✗ NO significativo'
    print(f'  {col:<12}  {rho:>+8.4f}  {pval:>10.4f}  {verdict}')


---
## SECCIÓN 5: Calidad de Jugadores por Equipo (players.csv)


In [ ]:
# ── EDA 4: Jugadores — ¿Qué equipos tienen más calidad FPL? ──
players.columns = players.columns.str.strip()

# Agregar métricas FPL por equipo
team_fpl = players.groupby('team').agg(
    jugadores   =('player_name', 'count'),
    total_pts   =('total_points', 'sum'),
    media_pts   =('total_points', 'mean'),
    total_threat=('threat', 'sum'),
    total_inf   =('influence', 'sum'),
).sort_values('total_pts', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
bars = ax.barh(team_fpl.index, team_fpl['total_pts'],
               color=PL_CYAN, edgecolor=PL_PURPLE)
ax.invert_yaxis()
ax.set_title('Total FPL Points por Equipo\n(Top 15)', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Puntos FPL acumulados (plantilla completa)')

ax = axes[1]
ax.scatter(team_fpl['total_threat'], team_fpl['total_inf'],
           s=team_fpl['total_pts']/5, color=PL_PINK,
           alpha=0.7, edgecolors=PL_PURPLE)
for team, row in team_fpl.iterrows():
    ax.annotate(team, (row['total_threat'], row['total_inf']),
                fontsize=7, ha='left', va='bottom')
ax.set_title('Threat vs Influence por Equipo\n(tamaño = FPL Points)', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Total Threat FPL')
ax.set_ylabel('Total Influence FPL')

plt.suptitle('EDA 4 — Calidad de Jugadores por Equipo (players.csv)', fontsize=14,
             fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo3_eda4_jugadores.png', bbox_inches='tight', dpi=150)
plt.show()


---
## SECCIÓN 6: Análisis de Disparos y Conversión xG (xg_train.csv)


In [ ]:
# ── EDA 5: xG y Eventos — ¿Qué equipos crean más peligro? ──
xg['is_goal_int'] = xg['is_goal'].astype(int)
team_xg = xg.groupby('team_name').agg(
    disparos=('is_goal','count'),
    goles   =('is_goal_int','sum'),
    big_ch  =('is_BigChance','sum'),
    penalties=('is_Penalty','sum'),
).assign(conv_rate=lambda d: d['goles']/d['disparos']*100).sort_values('disparos', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
x_pos = np.arange(len(team_xg))
w = 0.4
ax.bar(x_pos - w/2, team_xg['disparos'], width=w, color=PL_PURPLE, label='Disparos', alpha=0.85)
ax.bar(x_pos + w/2, team_xg['goles'],    width=w, color=PL_CYAN,   label='Goles',    alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(team_xg.index, rotation=45, ha='right', fontsize=8)
ax.set_title('Disparos y Goles por Equipo\n(xg_train.csv)', color=PL_PURPLE, fontweight='bold')
ax.legend()

ax = axes[1]
ax.scatter(team_xg['disparos'], team_xg['conv_rate'],
           c=team_xg['big_ch'], cmap='Purples', s=100, edgecolors=PL_PINK)
for team, row in team_xg.iterrows():
    ax.annotate(team, (row['disparos'], row['conv_rate']), fontsize=7)
ax.set_xlabel('Total Disparos')
ax.set_ylabel('Tasa de Conversión (%)')
ax.set_title('Disparos vs Tasa de Conversión\n(color = Big Chances)', color=PL_PURPLE, fontweight='bold')

plt.suptitle('EDA 5 — Análisis xG y Eventos (xg_train.csv)', fontsize=14,
             fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo3_eda5_xg.png', bbox_inches='tight', dpi=150)
plt.show()

print('Estadísticas de conversión por equipo (Top 10):')
print(team_xg[['disparos','goles','big_ch','conv_rate']].round(2).head(10).to_string())


---
## SECCIÓN 7: Cruce de los 3 Datasets

Correlaciones entre calidad FPL, rendimiento en partidos y eficacia ofensiva xG.


In [ ]:
# ── EDA 6: Resumen cruzado — los 3 datasets juntos ──
# Cruzar calidad FPL con rendimiento en matches
home_results = matches.groupby('home_team').apply(
    lambda g: pd.Series({
        'partidos_home': len(g),
        'pct_win_home' : (g['ftr']=='H').mean()*100,
        'goles_home'   : g['fthg'].mean(),
    })
).reset_index().rename(columns={'home_team':'team'})

fpl_team = players.groupby('team').agg(
    total_pts   =('total_points','sum'),
    total_threat=('threat','sum'),
).reset_index()

cross_df = home_results.merge(fpl_team, on='team', how='inner')
cross_df = cross_df.merge(
    team_xg[['disparos','conv_rate']].reset_index().rename(columns={'team_name':'team'}),
    on='team', how='left'
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(cross_df['total_pts'], cross_df['pct_win_home'],
           color=PL_CYAN, s=80, edgecolors=PL_PURPLE, alpha=0.8)
for _, row in cross_df.iterrows():
    ax.annotate(row['team'], (row['total_pts'], row['pct_win_home']), fontsize=6)
rho1, p1 = spearmanr(cross_df['total_pts'], cross_df['pct_win_home'])
ax.set_title(f'FPL Points vs % Victorias Locales\nSpearman ρ={rho1:.3f}, p={p1:.3f}',
             color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('FPL Points totales del equipo')
ax.set_ylabel('% Victorias en casa')

ax = axes[1]
ax.scatter(cross_df['total_threat'], cross_df['goles_home'],
           color=PL_PINK, s=80, edgecolors=PL_PURPLE, alpha=0.8)
for _, row in cross_df.iterrows():
    ax.annotate(row['team'], (row['total_threat'], row['goles_home']), fontsize=6)
rho2, p2 = spearmanr(cross_df['total_threat'], cross_df['goles_home'])
ax.set_title(f'FPL Threat vs Goles/Partido (local)\nSpearman ρ={rho2:.3f}, p={p2:.3f}',
             color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Total Threat FPL')
ax.set_ylabel('Goles/partido en casa')

ax = axes[2]
cross_plot = cross_df.dropna(subset=['disparos','pct_win_home'])
ax.scatter(cross_plot['disparos'], cross_plot['pct_win_home'],
           c=cross_plot['total_pts'], cmap='Purples', s=100, edgecolors=PL_PINK)
for _, row in cross_plot.iterrows():
    ax.annotate(row['team'], (row['disparos'], row['pct_win_home']), fontsize=6)
rho3, p3 = spearmanr(cross_plot['disparos'], cross_plot['pct_win_home'])
ax.set_title(f'Disparos Totales vs % Victorias Locales\nSpearman ρ={rho3:.3f}, p={p3:.3f}',
             color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Disparos totales (temporada)')
ax.set_ylabel('% Victorias en casa')

plt.suptitle('EDA 6 — Cruce de los 3 Datasets: FPL + Partidos + xG', fontsize=14,
             fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo3_eda6_cruce.png', bbox_inches='tight', dpi=150)
plt.show()

print('Correlaciones cruzadas (Spearman):')
print(f'  FPL Points → % Victorias locales : ρ={rho1:.3f}, p={p1:.4f}')
print(f'  FPL Threat → Goles/partido local : ρ={rho2:.3f}, p={p2:.4f}')
print(f'  Disparos   → % Victorias locales : ρ={rho3:.3f}, p={p3:.4f}')


---
## RESUMEN

| Análisis | Hallazgo principal |
|---|---|
| Distribución resultados | H≈46%, D≈26%, A≈28% — desbalance moderado |
| Factor árbitro | Sin efecto estadísticamente significativo (Kruskal-Wallis p>0.05) |
| Cuotas Bet365 | b365h y b365a sí correlacionan (p<0.001); b365d no (p=0.63) |
| Jugadores FPL | FPL Points correlaciona con victorias locales |
| xG disparos | Solo 11.2% de disparos son gol — fuerte desbalance de clase |
| Cruce datasets | FPL Threat y disparos correlacionan moderadamente con rendimiento |
